# G03 - Veri Edinme, Temizleme ve Zenginleştirme
## Hazırlayan: Adem Ataş (24430040022) - Veri Mühendisi

Bu çalışma defteri; İBB BELBİM saatlik ulaşım verilerinin (3 aylık) chunking yöntemiyle okunmasını, Open-Meteo saatlik hava durumu verisiyle eşleştirilmesini ve takvim değişkenlerinin türetilmesini içerir.

In [ ]:
import pandas as pd
from pathlib import Path
from src.veri import toplula, hava_cek, ozellik_uret

In [ ]:
# 1. 3 Aylık BELBİM Ham Verilerinin Chunksize ile Toplanması
aylar = ['temmuz2024', 'agustos2024', 'eylul2024']
ilce_listesi = []

for ay in aylar:
    print(f'{ay} verisi okunuyor...')
    ilce, _ = toplula(Path(f'../data/raw/belbim_{ay}.csv'))
    ilce_listesi.append(ilce)

df_ulasim = pd.concat(ilce_listesi, ignore_index=True)
df_ulasim = df_ulasim.groupby(['tarih', 'saat', 'mod', 'ilce'], as_index=False)[['yolcu', 'yolculuk']].sum()
print('Toplam Saatlik İlçe Kayıt Sayısı:', len(df_ulasim))
df_ulasim.head()

In [ ]:
# 2. Open-Meteo API ile Saatlik Hava Durumu Çekilmesi
df_hava = hava_cek('2024-07-01', '2024-09-30')
df_hava.head()

In [ ]:
# 3. Ulaşım ve Hava Verisinin Zaman Bazlı Birleştirilmesi (Merge)
df_ulasim['tarih_dt'] = pd.to_datetime(df_ulasim['tarih']) + pd.to_timedelta(df_ulasim['saat'], unit='h')
tarih_kol = 'dt' if 'dt' in df_hava.columns else df_hava.columns[0]
df_hava[tarih_kol] = pd.to_datetime(df_hava[tarih_kol])

df_birlesik = pd.merge(df_ulasim, df_hava, left_on='tarih_dt', right_on=tarih_kol, how='left')
df_birlesik.head()

In [ ]:
# 4. Takvim ve Dönem Özelliklerinin Türetilmesi
df_final = ozellik_uret(df_birlesik)

# Temiz Özet Veriyi Kaydetme
df_final.to_csv('ulasim_toplanmis_ozet.csv', index=False)
print('Veri başarıyla kaydedildi! Boyut:', df_final.shape)
df_final.head()